#### Objective problem: Với ngân sách khuyến mãi cố định, nên gửi voucher cho nhóm rider nào để tạo ra nhiều incremental trips nhất?

#### Mục lục

| Phần | Nội dung |
|---|---|
| **Tổng quan về bộ dữ liệu** | đơn vị quan sát, định nghĩa 14 cột kèm nguồn và cách tạo, nguồn dữ liệu |
| **Pipeline** | pipeline tạo synthetic data |
| **Causal contract** | unit, treatment, cửa sổ quan sát, potential outcome và cách gán treatment |
| **SET UP** | Thiết lập đường dẫn và các chỉ số |
| **1. Làm sạch TLC + tạo biến** | Đọc dữ liệu đã làm sạch ở bước EDA và tạo feature |
| **2. Gán archetype cho từng chuyến** | 16 nhóm = 4 khung giờ × 4 cự ly, đo tỷ trọng thật `p_arch` của từng nhóm |
| **3. Sinh khung thời gian của rider** | từ `lambda` → `active_days` → `total_rides` và `recency_days` |
| **4. Gán chuyến từ TLC cho từng rider** | mỗi rider có hành vi khác nhau `w_i ~ Dirichlet`, bốc chuyến thật theo hành vi đó |
| **4.1 Tổng hợp chuyến → 14 cột đặc trưng** | `aggregate()` tính 9 cột hành vi từ chuyến đã bốc; `age`, `gender` lấy từ Kaggle |
| **5. Sinh outcome + gán treatment** | dựng ba confounder, các cột của tầng thí nghiệm |
| **5.1 Giả định `cate_true` / `ite_realized` + kết quả nền** | tách phần kỳ vọng khỏi nhiễu cá nhân |
| **5.2 Chia treatment + suy ra outcome** | RCT bốc 50/50 trong từng khối; `Y = nền + ite_realized · T` |
| **6. Xuất kết quả** | Xuất dữ liệu |

#### Tổng quan về bộ dữ liệu 

**1. Dataset Objective**

Mục tiêu của việc sinh dữ liệu synthetic là xây dựng một rider-level dataset mô phỏng dữ liệu khách hàng nhằm phục vụ:

- Customer segmentation để xác định các nhóm rider có hành vi khác nhau.
- Targeting promotion để tìm nhóm khách hàng có khả năng tạo ra incremental trips cao.
- A/B testing.

**2. Data Sources**

Bộ dữ liệu synthetic được tạo ra dựa trên sự kết hợp của 2 bộ dữ liệu NYC TLC Trip Data và đặc điểm người dùng từ ride-sharing platform.

| Nguồn |  | Giải thích |
|---|---|---|
| **NYC TLC Trip Records** | 9 cột hành vi (cột 4, 7–14) | hành vi thật ở **cấp chuyến**: giờ giấc, cự ly, chi tiêu, tip, vùng di chuyển. TLC ẩn danh nên không có định danh khách hàng — đó là lý do phải tự dựng rider |
| **Ride-Sharing Platform Data** (Kaggle) | `age`, `gender` | nhân khẩu học mà TLC không cung cấp |
| **Giả định synthetic** | `total_rides`, `recency_days`, và toàn bộ tầng thí nghiệm | mô phỏng tần suất, trạng thái ngủ đông, treatment và outcome |

**3. Unit of Observation**

Dataset được xây dựng ở cấp độ customer/rider level.
Mỗi dòng dữ liệu đại diện cho: Một rider duy nhất trong hệ thống ride-hailing platform trong khoảng thời gian quan sát.

| # | Cột | Kiểu | Đơn vị | Ý nghĩa | Nguồn | Cách tạo |
|---|---|---|---|---|---|---|
| 1 | `user_id` | int64 | mã | mã định danh rider, khoá chính `1` → `20000` | tự sinh | `np.arange(1, 20001)` |
| 2 | `age` | int16 | năm | tuổi của rider | Kaggle | quantile matching từ `users.csv` |
| 3 | `gender` | object | phân loại | `Female` / `Male` / `Other` | Kaggle | bốc theo tần suất bộ gốc |
| 4 | `home_borough` | object | phân loại | quận hoạt động chính | TLC | zone đón khách nhiều nhất → quận của zone đó |
| 5 | `total_rides` | int32 | chuyến | số chuyến trong 90 ngày | **giả định** | `Poisson(lambda · a_days / 30)`, sàn 1 |
| 6 | `recency_days` | int16 | ngày | số ngày kể từ chuyến gần nhất | **giả định** | `(90 − a_days) + Exp(30/lambda)`, trần 90 |
| 7 | `typical_distance` | float64 | dặm | cự ly chuyến đi điển hình | TLC | median `trip_distance` |
| 8 | `route_entropy` | float64 | nat | độ đa dạng tuyến đi; **0** = chỉ đi một tuyến duy nhất | TLC | entropy Shannon trên cặp `PU→DO` |
| 9 | `pct_airport` | float64 | 0–1 | tỷ lệ chuyến sân bay | TLC | tỷ lệ `is_air` |
| 10 | `pref_time_bucket` | object | phân loại | khung giờ hay dùng nhất | TLC | khung giờ đông chuyến nhất |
| 11 | `weekend_ratio` | float64 | 0–1 | tỷ lệ chuyến vào cuối tuần | TLC | tỷ lệ `is_wknd` |
| 12 | `pct_flex_payment` | float64 | 0–1 | tỷ lệ chuyến trả bằng Flex Fare | TLC | tỷ lệ `payment_type == 0` |
| 13 | `pct_tip_rate` | float64 | tỷ lệ | tỷ lệ tip | TLC | TB `tip_rate`, **chỉ trên chuyến trả thẻ** |
| 14 | `avg_fare` | float64 | USD/chuyến | cước trung bình mỗi chuyến | TLC | mean `fare_amount` |


#### Pipeline

Hai nhánh chạy song song rồi gặp nhau ở bước 4: một nhánh là hành vi chuyến đi, nhánh kia là rider đi bao nhiêu chuyến

```
            NYC TLC                             Giả định
                                           BASELINE_MEAN, CV_LAMBDA,
              |                           ZERO_TARGET, WAKE_RATE
              v                                     |
  1. Làm sạch + tạo biến                            v
                                         3. Sinh khung thời gian
                                            (total_rides, recency)
              |                                     |
              v                                     |
  2. Gán archetype                                  |
     16 ô = 4 giờ x 4 cự ly                         |
              |                                     |
              v                                     |
           p_arch                                   |
              |                                     |
              |     KIỂU chuyến        SỐ chuyến    |
              +---------------+   +-----------------+
                              v   v
                4. Gán chuyến từ TLC cho từng rider
                   bốc đúng total_rides chuyến thật,
                   theo w ~ Dirichlet(ALPHA * p_arch)
                              |
                              v
                4.1 Tổng hợp -> 9 cột HÀNH VI
                    median cự ly, mean cước, % sân bay,
                    % cuối tuần, % Flex, tip rate,
                    entropy tuyến, khung giờ trội, quận
                              |
                              v
                    14 cột = 9 hành vi (trên)
                           + total_rides, recency (bước 3)
                           + age, gender (Kaggle)
                           + user_id
                              |
                              v
                        Sinh outcome + gán treatment
```

#### Causal contract

| | |
|---|---|
| **Unit** | rider *i*, `i = 1 … 20.000`|
| **Treatment** | `T_i ∈ {0, 1}`  |
| **Feature window** | 90 ngày |
| **Outcome** | `Y_i` = số chuyến trong **30 ngày**  |
| **Potential outcomes** | `Y_i(0)`, `Y_i(1)`|
| **Estimand** | `ate_true = mean(cate_true)` — tác động kỳ vọng theo covariate · `ate_realized = mean(ite_realized)` — tác động thực tế trên đúng mẫu này |
| **Observed outcome** | `Y_rct ` · `Y_obs ` |

## SET UP

- N_RIDERS: Số lượng rider giả lập được tạo ra.
- Seed cho bộ sinh số ngẫu nhiên.
- alpha: Điều chỉnh mức độ hành vi của rider.

In [1]:
import os
import numpy as np, pandas as pd
from scipy.stats import nbinom, norm, ks_2samp, binomtest
from scipy.optimize import brentq
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

B   = r"c:/Users/Linh/Desktop/Growth & Experimentation Project for Ride-Hailing Promotions"
KAG = os.path.expanduser("~/.cache/kagglehub/datasets/adnananam/"
                         "ride-sharing-platform-data/versions/1")
OUTDIR = os.path.join(B, "02. Synthetic_data", "output")
os.makedirs(OUTDIR, exist_ok=True)

N_RIDERS = 20_000
SEED     = 42
ALPHA    = 0.1    # do tap trung archetype cua rider
rng = np.random.default_rng(SEED)
print("TLC   :", os.path.isdir(os.path.join(B, "data")))
print("Kaggle:", os.path.isdir(KAG))

TLC   : True
Kaggle: True


## 1. Làm sạch TLC + tạo biến

In [2]:
cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
        "fare_amount", "tip_amount", "payment_type", "RatecodeID",
        "PULocationID", "DOLocationID", "Airport_fee"]

# Đọc dữ liệu chuyến đi từ file data đã làm sạch ở bước EDA
d = pd.read_csv(f"{B}/data/yellow_tripdata_2026.csv", usecols=cols,
                parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"]
                ).rename(columns={"Airport_fee": "airport_fee"})
n_raw = len(d)

# Tính thời lượng chuyến đi từ cột tpep_pickup_datetime và tpep_dropoff_datetime
d["dur"] = (d.tpep_dropoff_datetime - d.tpep_pickup_datetime).dt.total_seconds() / 60
d = d[(d.fare_amount > 2.5) & (d.dur > 1)].reset_index(drop=True)

# ty le tip chi do duoc tren chuyen the (TLC khong ghi tip tien mat)
d["tip_rate"] = np.where(d.payment_type == 1,
                         np.clip(d.tip_amount / d.fare_amount, 0, 1), np.nan)

# Tạo các biến đặc trưng (feature engineering) từ dữ liệu chuyến đi
# hour: giờ bắt đầu chuyến từ cột tpep_pickup_datetime
d["hour"]    = d.tpep_pickup_datetime.dt.hour.astype(np.int8)

# is_wknd: chuyến đi cuối tuần từ cột tpep_pickup_datetime
d["is_wknd"] = (d.tpep_pickup_datetime.dt.dayofweek >= 5).astype(np.int8)

# is_air: chuyến đi sân bay.
AIRPORT_ZONES = [1, 132, 138]                       # Newark, JFK, LaGuardia
d["is_air"]  = (d.RatecodeID.isin([2, 3])
                | (d.airport_fee.fillna(0) > 0)
                | d.PULocationID.isin(AIRPORT_ZONES)
                | d.DOLocationID.isin(AIRPORT_ZONES)).astype(np.int8)

# is_flex: thanh toán linh hoạt từ cột payment_type 
d["is_flex"] = (d.payment_type == 0).astype(np.int8)

# is_card: thanh toán bằng thẻ từ cột payment_type
d["is_card"] = (d.payment_type == 1).astype(np.int8)

## 2. Gán archetype cho từng chuyến

Mục đích bước này gán archetype cho mỗi chuyến dựa trên thời gian và khoảng cách 

**16 archetype = 4 khung giờ × 4 nhóm cự ly.**

| | |
|---|---|
| Khung giờ | `morning` 6–10h · `midday` 10–16h · `evening` 16–22h · `late_night` 22–6h |
| Cự ly | `short` < 2 dặm · `medium` 2–5 dặm · `long` 5–10 dặm · `very_long` > 10 dặm |



In [3]:
# Chia khung giờ trong ngày thành 4 nhóm: morning | midday | evening | late_night
def time_bucket(hour):
    if 6 <= hour < 10:    return "morning"
    elif 10 <= hour < 16: return "midday"
    elif 16 <= hour < 22: return "evening"
    else:                 return "late_night"

DIST_EDGE  = [2, 5, 10]                                   # short | medium | long | very_long
DIST_LABEL = ["short", "medium", "long", "very_long"]

BUCKET = {h: time_bucket(h) for h in range(24)}

# Gán time_bucket cho từng chuyến đi dựa trên giờ đón khách
d["time_bucket"]     = d.hour.map(BUCKET)

# Chia nhóm khoảng cách chuyến đi thành 4 nhóm: short | medium | long | very_long
d["distance_bucket"] = pd.Categorical.from_codes(
    np.searchsorted(DIST_EDGE, d.trip_distance.to_numpy(), side="right"), DIST_LABEL)

# Tạo archetype với 4 khung giờ x 4 nhóm khoảng cách = 16 archetype
d["archetype"]       = d.time_bucket.astype(str) + "_" + d.distance_bucket.astype(str)

# Danh sách archetype và số lượng archetype
ARCH   = sorted(d.archetype.unique())
N_ARCH = len(ARCH)
d["arch"] = d.archetype.map({a: i for i, a in enumerate(ARCH)}).astype(np.int8)

# Tỷ lệ archetype trong dữ liệu
p_arch = d.arch.value_counts(normalize=True).reindex(range(N_ARCH)).to_numpy()

# Thống kê đặc điểm từng archetype
# Với mỗi archetype tính: tỷ trọng trong dữ liệu, khoảng cách trung vị, fare trung vị, % chuyến sân bay, % chuyến Flex, tip rate trung bình
chk = pd.DataFrame({
    "ty trong":    p_arch,
    "dist median": [np.median(d.trip_distance.values[d.arch == k]) for k in range(N_ARCH)],
    "fare median": [np.median(d.fare_amount.values[d.arch == k]) for k in range(N_ARCH)],
    "% san bay":   [d.is_air.values[d.arch == k].mean()*100 for k in range(N_ARCH)],
    "% Flex":      [d.is_flex.values[d.arch == k].mean()*100 for k in range(N_ARCH)],
    "tip rate":    [d.tip_rate.values[(d.arch == k) & (d.is_card == 1)].mean() for k in range(N_ARCH)],
}, index=ARCH).round(3)
print(f"{N_ARCH} archetype = 4 khung gio x {len(DIST_LABEL)} nhom cu ly:")
display(chk)

16 archetype = 4 khung gio x 4 nhom cu ly:


,ty trong,dist median,fare median,% san bay,% Flex,tip rate
evening_long,0.036,6.90,34.50,24.164,36.123,0.200
evening_medium,0.104,2.82,19.80,1.277,31.045,0.231
evening_short,0.204,1.10,10.00,0.191,17.848,0.299
evening_very_long,0.025,16.24,70.00,72.786,10.684,0.176
late_night_long,0.030,6.84,32.27,16.570,55.545,0.202
late_night_medium,0.065,3.05,19.61,1.456,49.456,0.228
late_night_short,0.080,1.20,10.00,0.239,33.174,0.288
late_night_very_long,0.016,14.40,56.20,56.232,29.581,0.168
midday_long,0.031,7.05,35.20,26.529,31.900,0.166
midday_medium,0.078,2.80,21.20,1.187,29.228,0.206


## 3. Sinh khung thời gian của rider

Mục đích bước này là mô phỏng hành vi của rider trong 90 ngày với các đặc trưng total_rides và recency_days 

Toàn bộ bước này bắt nguồn từ biến `lambda_i` = số chuyến kỳ vọng của rider *i* trong 30 ngày. Mọi thứ khác suy ra từ đó:

```
lambda_i ──> ngủ đông? ──> a_days ──┬──> total_rides
                                └──> recency_days
```
#### Giả định 

**`BASELINE_MEAN = 8`** — số chuyến kỳ vọng / rider / 30 ngày

Tương đương **~2 chuyến mỗi tuần**. 

TLC có ~3,48 triệu chuyến yellow taxi mỗi tháng (10.451.513 ÷ 3), nên:

$$\text{quần thể} = \frac{3.483.838}{8} \approx 435.480 \text{ người}$$

| `BASELINE_MEAN` | Quần thể suy ra | |
|---|---|---|
| 2 | 1.741.919 | quá rộng cho riêng yellow taxi |
| 4 | 870.959 | chấp nhận được |
| **8** | **435.480** | **chọn** |
| 12 | 290.320 | chấp nhận được |
| 20 | 174.192 | chỉ còn nhóm đi rất nhiều |



**`CV_LAMBDA = 0.60`** — chênh lệch tần suất giữa các rider

**`ZERO_TARGET = 0.20`** — khoảng 20% rider không phát sinh chuyến nào trong 30 ngày tới. Số 0 đến từ hai nguồn khác nhau: đã ngừng dùng dịch vụ và vẫn hoạt động nhưng tháng đó tình cờ không đi 

**`WAKE_RATE = 0.25`** — voucher đánh thức bao nhiêu % người ngủ đông

**Khung thời gian:** `W_FEAT = 90` ngày quan sát đặc trưng, `W_OUT = 30` ngày đo outcome ngay sau đó.

| Bước | Công thức | Ý nghĩa |
|---|---|---|
| 1 | `lambda_i ~ LogNormal(E=8, CV=0.6)` | tần suất của rider, luôn dương  |
| 2 | `dorm_i ~ Bernoulli(pi_i)`, `logit(pi_i) = c − 0.9·z(log lambda_i)` | ai đã ngừng dùng dịch vụ - người đi ít dễ ngừng hơn |
| 3 | `a_days = 90` nếu còn hoạt động, `~ U(5, 70)` nếu đã ngừng | ngày cuối cùng còn hoạt động |
| 4 | `total_rides ~ Poisson(lambda · a_days / 30)`, sàn 1 | chuyến **quan sát được** |
| 5 | `recency = (90 − a_days) + Exp(30/lambda)` , trần 90 | thời gian kể từ chuyến cuối |


Kết quả: rider còn hoạt động có recency trung bình ~4,6 ngày, rider ngủ đông ~59 ngày. Nhờ cùng sinh từ `lambda`, `total_rides` và `recency_days` tương quan mạnh 


In [ ]:
NB_DISPERSION = 4.0      # r cua negative binomial; nho -> duoi day

CV_LAMBDA     = 0.60   # chênh lệch tần suất chuyến đi giữa các rider 
BASELINE_MEAN = 8.0    # số chuyến đi trung bình trong 30 ngày của rider trung bình
ZERO_TARGET   = 0.20   # tỷ lệ người dùng không có chuyến đi nào trong 30 ngày 
WAKE_RATE     = 0.25   # tỷ lệ người dùng bị dorm nhưng được voucher đánh thức trong 30 ngày
W_FEAT, W_OUT = 90, 30  # số ngày trong cửa sổ quan sát và số ngày quan sát

# Sinh nhu cầu đi xe cho từng rider. Mỗi rider có mức độ sử dụng khác nhau, mô phỏng sự khác biệt hành vi giữa người dùng thực tế.
# Sử dụng LogNormal distribution vì số chuyến luôn dương
sig  = np.sqrt(np.log(1 + CV_LAMBDA ** 2))
lam  = BASELINE_MEAN * np.exp(sig * rng.standard_normal(N_RIDERS) - sig ** 2 / 2)
z_lam = (np.log(lam) - np.log(lam).mean()) / np.log(lam).std()

# Tạo xu hướng ngủ đông dựa trên nhu cầu sử dụng:
# lambda thấp -> z_lam thấp -> xác suất dormant cao hơn
lin_d = -0.90 * z_lam
# Xác suất Negative Binomial sinh ra 0 chuyến 
p_nb0 = (NB_DISPERSION / (NB_DISPERSION + lam)) ** NB_DISPERSION
def p_zero(c):
    pi = 1 / (1 + np.exp(-(c + lin_d)))
    return (pi + (1 - pi) * p_nb0).mean()
c_d  = brentq(lambda c: p_zero(c) - ZERO_TARGET, -20, 20)
pi_d = 1 / (1 + np.exp(-(c_d + lin_d)))
# Sinh trạng thái dormant 
dorm = rng.random(N_RIDERS) < pi_d

# voucher danh thuc ai: boc DOC LAP voi dorm 
woken = rng.random(N_RIDERS) < WAKE_RATE

# Xác định số ngày cuối cùng rider còn hoạt động trong cửa sổ feature.
# - Rider active: hoạt động toàn bộ 90 ngày.
# - Rider dormant: ngừng hoạt động trước khi kết thúc cửa sổ.
a_days = np.where(dorm, rng.uniform(5, W_FEAT - 20, N_RIDERS), float(W_FEAT))

# Sinh số chuyến trong feature window dựa trên lambda và số ngày hoạt động
n_i = np.maximum(rng.poisson(lam * a_days / W_OUT), 1)

# Tính recency_days: thời gian kể từ lần hoạt động gần nhất
rec = np.minimum((W_FEAT - a_days) + rng.exponential(W_OUT / lam), W_FEAT).round().astype(int)

print(f"lambda        : TB {lam.mean():.2f} chuyen/30 ngay | CV {lam.std()/lam.mean():.2f}")
print(f"ngu dong      : {dorm.mean():.1%}  (muc tieu ty le Y=0 la {ZERO_TARGET:.0%})")
print(f"  trong do bi voucher danh thuc: {(dorm & woken).sum():,} "
      f"({(dorm & woken).sum()/dorm.sum():.1%}, dat WAKE_RATE = {WAKE_RATE:.0%})")
print(f"total_rides   : TB {n_i.mean():.1f} | trung vi {np.median(n_i):.0f} | max {n_i.max()}"
      f"   ({W_FEAT} ngay)")
print(f"recency_days  : TB {rec.mean():.1f} | trung vi {np.median(rec):.0f} | max {rec.max()}")
print(f"   con hoat dong: TB {rec[~dorm].mean():.1f} ngay")
print(f"   da ngu dong  : TB {rec[dorm].mean():.1f} ngay")

lambda        : TB 8.03 chuyen/30 ngay | CV 0.60
ngu dong      : 17.6%  (muc tieu ty le Y=0 la 20%)
  trong do bi voucher danh thuc: 890 (25.3%, dat WAKE_RATE = 25%)
total_rides   : TB 22.4 | trung vi 19 | max 203   (90 ngay)
recency_days  : TB 14.2 | trung vi 4 | max 90
   con hoat dong: TB 4.6 ngay
   da ngu dong  : TB 58.8 ngay


## 4. Gán chuyến từ TLC cho từng rider

Số chuyến đã có ở bước 3. Bước này quyết định **rider đó đi những chuyến nào**, bằng cách bốc chuyến từ dữ liệuTLC.

Giả định mỗi rider có một **hành vi archetype**:

```
w_i ~ Dirichlet(ALPHA · p_arch · N_ARCH)
```

`p_arch` là tỷ trọng 16 archetype trong TLC và một chuyến TLC có thể được gán cho nhiều rider. Mục đích là **mượn phân phối thật** của cự ly, cước, giờ giấc, vùng đón trả.

**Kết quả:** 448.422 chuyến được gán cho 20.000 rider — trung bình 22,4 chuyến mỗi rider trong 90 ngày.

In [5]:
# Sinh phân phối archetype cho từng rider bằng Dirichlet.
# Mỗi rider có tỷ lệ hành vi khác nhau trên N_ARCH nhóm. ALPHA lớn -> rider giống phân phối chung TLC hơn. ALPHA nhỏ -> hành vi giữa rider khác biệt hơn.
w_i = rng.dirichlet(ALPHA * p_arch * N_ARCH, size=N_RIDERS)
rid_all = np.repeat(np.arange(N_RIDERS), n_i)

# Chọn archetype cho từng chuyến dựa trên profile hành vi của rider.
# Rider có xác suất archetype nào cao thì chuyến của họ có khả năng thuộc archetype đó cao hơn.
a_all = np.clip((np.cumsum(w_i, 1)[rid_all] < rng.random(len(rid_all))[:, None]).sum(1),
                0, N_ARCH - 1)

# Chuyển các thuộc tính chuyến TLC sang numpy array
arch  = d.arch.to_numpy()
dist  = d.trip_distance.to_numpy(np.float64); fare = d.fare_amount.to_numpy(np.float64)
hour  = d.hour.to_numpy(np.int64)
is_air = d.is_air.to_numpy(np.int8);   is_wknd = d.is_wknd.to_numpy(np.int8)
is_flex = d.is_flex.to_numpy(np.int8); is_card = d.is_card.to_numpy(np.int8)
trate = np.nan_to_num(d.tip_rate.to_numpy(np.float64))
PU = d.PULocationID.to_numpy(np.int64); DO = d.DOLocationID.to_numpy(np.int64)
N_ZONE = 266

# boc DEU trong tung archetype
ord_a = np.argsort(arch, kind="stable").astype(np.int32)
cnt_a = np.bincount(arch, minlength=N_ARCH)
st_a  = np.concatenate([[0], np.cumsum(cnt_a)])
idx_all = ord_a[st_a[a_all] +
                (rng.random(len(a_all)) * cnt_a[a_all]).astype(np.int64)].astype(np.int64)

print(f"{N_RIDERS:,} rider / {len(idx_all):,} chuyen "
      f"(trung binh {len(idx_all)/N_RIDERS:.1f} chuyen/rider trong {W_FEAT} ngay)")

20,000 rider / 448,422 chuyen (trung binh 22.4 chuyen/rider trong 90 ngay)


### 4.1 Tổng hợp chuyến → 14 cột đặc trưng

Hàm `aggregate()` nhận các chuyến kèm nhãn rider rồi tính ra chân dung từng người. Chín cột hành vi **không được sinh ra** — chúng được **tính** từ các chuyến đã bốc.

| Cột | Công thức trên các chuyến của rider |
|---|---|
| `total_rides` | **= `n_i` của bước 3** |
| `typical_distance` | **median** `trip_distance`  |
| `avg_fare` | **mean** `fare_amount` |
| `pct_airport` | tỷ lệ chuyến sân bay |
| `weekend_ratio` | tỷ lệ chuyến cuối tuần |
| `pct_flex_payment` | tỷ lệ chuyến trả Flex |
| `pct_tip_rate` | TB `tip_rate` **chỉ trên chuyến trả thẻ** — TLC không ghi tip tiền mặt |
| `pref_time_bucket` | khung giờ có nhiều chuyến nhất (`H.argmax`) |
| `home_borough` | **zone** đón khách nhiều nhất, rồi tra ra quận (`Z.argmax`) |
| `route_entropy` | $-\sum p \ln p$ trên các cặp `PU→DO` |


**Hai cột đến thẳng từ bước 3**:

- `total_rides` = `n_i`
- `recency_days` = `rec`

Hai cột lấy từ nguồn **Ride-Sharing Platform Data** (Kaggle):
- `age`, `gender` lấy phân phối bằng quantile matching — giữ đúng phân phối tuổi và tỷ lệ giới tính

**Kết quả:** `riders` — bảng 20.000 dòng × 14 cột hoàn chỉnh, đầu vào cho bước 5 và cho bước 03 (segmentation).

In [ ]:
# Mapping LocationID của TLC sang Borough
zl = pd.read_csv(f"{B}/data/taxi_zone_lookup.csv")
bmap = dict(zip(zl.LocationID, zl.Borough.fillna("Unknown")))

def aggregate(rid, tr):
    """Tong hop chuyen -> dac trung rider."""
    n   = N_RIDERS
    # Đếm số chuyến của mỗi rider
    # => total_rides
    cnt = np.bincount(rid, minlength=n)
    g   = pd.DataFrame({"r": rid, "dist": tr["dist"], "fare": tr["fare"]})

    # Tính khoảng cách điển hình bằng median
    med  = g.groupby("r").dist.median().reindex(range(n)).to_numpy()

    # Tính giá cước trung bình của rider
    avgf = g.groupby("r").fare.mean().reindex(range(n)).to_numpy()

    # Hàm tính tỷ lệ trung bình của các biến nhị phân
    rmean = lambda x: np.bincount(rid, weights=x.astype(float), minlength=n) / cnt

    # Tính tỷ lệ tip
    card   = tr["is_card"].astype(bool)                 # tip chi do duoc tren chuyen the
    n_card = np.bincount(rid[card], minlength=n)
    s_card = np.bincount(rid[card], weights=tr["trate"][card], minlength=n)

    # Tính route entropy: Đo mức độ đa dạng tuyến đường PU -> DO
    # Entropy thấp: rider thường đi cùng một vài tuyến cố định. Entropy cao: rider đi nhiều tuyến khác nhau
    route = tr["PU"] * 1000 + tr["DO"]                  # entropy Shannon tren cap PU->DO
    c = pd.Series(np.ones(len(rid))).groupby([rid, route]).size()
    p = c / c.groupby(level=0).transform("sum")
    ent = (-(p * np.log(p))).groupby(level=0).sum().reindex(range(n)).fillna(0).to_numpy()

    # Tạo histogram theo giờ và pickup zone
    # Dùng để tìm:
    # - giờ đi phổ biến nhất
    # - khu vực bắt xe phổ biến nhất
    H = np.zeros((n, 24), np.int32);     np.add.at(H, (rid, tr["hour"]), 1)
    Z = np.zeros((n, N_ZONE), np.int32); np.add.at(Z, (rid, tr["PU"]), 1)

    return pd.DataFrame({
        # số chuyến
        "total_rides":      cnt,

        # hành vi di chuyển
        "typical_distance": med,
        "route_entropy":    ent,

        # đặc điểm chuyến đi
        "pct_airport":      rmean(tr["is_air"]),
        "weekend_ratio":    rmean(tr["is_wknd"]),
        "pct_flex_payment": rmean(tr["is_flex"]),
        "pct_tip_rate":     np.where(n_card > 0, s_card / np.maximum(n_card, 1), 0.0),
        "avg_fare":         avgf,

        # thời gian và khu vực
        "pref_time_bucket": np.array([BUCKET[h] for h in H.argmax(1)], dtype=object),
        "home_borough":     np.array([bmap.get(int(z), "Unknown") for z in Z.argmax(1)],
                                     dtype=object),
    })

trips = dict(dist=dist[idx_all], fare=fare[idx_all], is_air=is_air[idx_all],
             is_wknd=is_wknd[idx_all], is_flex=is_flex[idx_all], is_card=is_card[idx_all],
             trate=trate[idx_all], hour=hour[idx_all], PU=PU[idx_all], DO=DO[idx_all])
riders = aggregate(rid_all, trips)
riders["recency_days"] = rec                    # tu buoc 3

# age va gender dựa theo bộ dữ liệu Ride-Sharing Platform Data 
# Đọc file users.csv từ Ride-Sharing Platform Data 
ku = pd.read_csv(f"{KAG}/users.csv", parse_dates=["registration_date"])

# Xây dựng phân phối tuổi
Q = np.linspace(0, 1, 1001)
q_age = np.quantile(ku.age, Q)

# Tính tỷ lệ giới tính 
pmf_g = ku.gender.value_counts(normalize=True)

# Sinh tuổi theo phân phối tuổi 
riders["age"] = np.interp(rng.random(N_RIDERS), Q, q_age).round().clip(18, 65).astype(int)

# Sinh giới tính theo tỷ lệ trong dữ liệu
riders["gender"] = rng.choice(pmf_g.index, N_RIDERS, p=pmf_g.values)

# Chuẩn hoá kiểu dữ liệu và làm tròn các biến số
riders["total_rides"] = riders.total_rides.astype(int)
for c in ["typical_distance", "avg_fare"]:
    riders[c] = riders[c].round(3)
for c in ["route_entropy", "pct_airport", "weekend_ratio", "pct_flex_payment", "pct_tip_rate"]:
    riders[c] = riders[c].clip(0, 1 if c != "route_entropy" else None).round(6)

COLS = ["user_id", "age", "gender", "home_borough", "total_rides", "recency_days",
        "typical_distance", "route_entropy", "pct_airport", "pref_time_bucket",
        "weekend_ratio", "pct_flex_payment", "pct_tip_rate", "avg_fare"]

# Tạo user_id cho từng rider
riders.insert(0, "user_id", np.arange(1, N_RIDERS + 1))
riders = riders[COLS]
assert len(COLS) == 14

NUM = ["total_rides", "typical_distance", "route_entropy", "pct_airport", "weekend_ratio",
       "pct_flex_payment", "pct_tip_rate", "avg_fare", "age", "recency_days"]

riders.to_csv(os.path.join(OUTDIR, "customer_features_final.csv"), index=False)
display(riders.head())

,user_id,age,gender,home_borough,total_rides,recency_days,typical_distance,route_entropy,pct_airport,pref_time_bucket,weekend_ratio,pct_flex_payment,pct_tip_rate,avg_fare
0,1,19,Male,Manhattan,25,3,1.20,3.218876,0.000000,evening,0.360000,0.160000,0.271971,11.293
1,2,28,Other,Queens,13,1,3.01,2.564949,0.307692,morning,0.230769,0.230769,0.154094,42.771
2,3,30,Female,Manhattan,24,2,0.98,3.062529,0.125000,evening,0.083333,0.083333,0.323680,14.534
3,4,20,Male,Queens,34,2,3.27,3.526361,0.176471,midday,0.411765,0.264706,0.201018,30.315
4,5,42,Female,Manhattan,11,7,1.33,2.397895,0.000000,midday,0.363636,0.181818,0.262834,15.875


In [7]:
display(riders.describe().round(3))

,user_id,age,total_rides,recency_days,typical_distance,route_entropy,pct_airport,weekend_ratio,pct_flex_payment,pct_tip_rate,avg_fare
count,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000
mean,10000.500,41.304,22.421,14.163,2.861,2.795,0.078,0.295,0.265,0.242,21.331
std,5773.647,13.967,16.173,22.795,2.743,0.873,0.133,0.155,0.162,0.063,9.880
min,1.000,18.000,1.000,0.000,0.100,0.000,0.000,0.000,0.000,0.000,3.700
25%,5000.750,29.000,11.000,1.000,1.300,2.398,0.000,0.200,0.167,0.213,14.332
50%,10000.500,41.000,19.000,4.000,1.868,2.944,0.000,0.286,0.250,0.247,19.020
75%,15000.250,54.000,30.000,12.000,2.975,3.367,0.105,0.374,0.348,0.278,25.218
max,20000.000,65.000,203.000,90.000,45.505,5.212,1.000,1.000,1.000,1.000,182.710


## 5. Sinh outcome + gán treatment

Bốn bước, theo đúng thứ tự:

| | Bước | Kết quả |
|---|---|---|
| 1 | Giả định tác động của voucher | cột `cate_true` (kỳ vọng theo covariate) và `ite_realized` (có thêm nhiễu cá nhân) |
| 2 | Tính **kết quả nền** — số chuyến khi *không* có voucher | `y_base` (biến cục bộ) |
| 3 | Chia treatment: **`T_rct`** bốc thăm · **`T_obs`** theo propensity | mục 5.1 và 5.2 |
| 4 | Suy ra outcome: **nền + `ite_realized`** nếu nhận voucher | cột `Y_rct`, `Y_obs` |

Metrics:

| Giả định | Ý nghĩa |
|---|---|
| `TAU_BASE = 1.5` | nền chung |
| `+ D_SHORT · z(−typical_distance)` | chuyến ngắn co giãn theo giá nhiều hơn |
| `+ D_MIDDAY` | rider trội khung giữa ngày linh hoạt về thời gian |
| `− P_MORNING` | rider giờ cao điểm sáng đi làm, gần như không co giãn |
| `− P_AIRPORT · z(pct_airport)` | chuyến sân bay là nhu cầu bắt buộc |
| `+ D_PRICE · z(−pct_tip_rate)` | người tip thấp nhạy giá hơn |
| `+ D_FLEX · z(pct_flex_payment)` | **nhạy giá bộc lộ qua hành vi** |
| `+ TAU_NOISE · N(0,1)` | nhiễu cá nhân — **chỉ có trong `ite_realized`**; `cate_true` là phần còn lại, tức kỳ vọng theo covariate |


**Ba confounder** dùng cho cả `mu0` lẫn `propensity_true` — vừa ảnh hưởng việc được gửi voucher vừa ảnh hưởng số chuyến:

- `z_prior` = `z(log1p(total_rides))` — đi nhiều
- `z_urban` = `z(home_borough == "Manhattan")` — ở nội đô
- `z_price` = `z(−pct_tip_rate)` — tip thấp, nhạy giá
- `z_air` = `z(pct_airport)`
- `z_flex` = `z(pct_flex_payment)` 

In [ ]:
# He so cua tang thi nghiem 
B_PRIOR, B_URBAN, B_PRICE, B_AIRPORT     = 0.35, 0.20, 0.10, 0.25   # -> mu0  (NEN)
# Hệ số của Individual Treatment Effect (ITE).
TAU_BASE,  D_SHORT,   D_MIDDAY           = 1.5,  1.0,  0.6          # -> cate_true / ite_realized
P_MORNING, P_AIRPORT, D_PRICE, TAU_NOISE = 0.5,  0.5,  0.6,  0.25
D_FLEX                                   = 0.45   # chon Flex Fare = tu bo lo minh nhay gia
# Tạo propensity score để sinh observational treatment có bias
TARGETING  = {"prior": 0.85, "urban": 0.60, "price": 0.45}          # -> propensity (OBS)
TREAT_RATE = 0.50

df = riders.copy()

# Chuẩn hoá dữ liệu
def zs(x):
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std()


def arms(y, t):
    """Trung binh nhanh treatment, nhanh control, va hieu so tho."""
    a, b = y[t == 1].mean(), y[t == 0].mean()
    return a, b, a - b


# Ba confounder (z_prior, z_urban, z_price) + ba bien chi vao cong thuc tac dong/mu0
df["is_urban"] = df.home_borough.eq("Manhattan").astype(np.int8)
df["z_prior"]  = zs(np.log1p(df.total_rides))      # di nhieu
df["z_urban"]  = zs(df.is_urban)                   # o noi do
df["z_price"]  = zs(-df.pct_tip_rate)              # tip thap -> nhay gia
df["z_air"]    = zs(df.pct_airport)                # hay ra san bay
df["z_dist"]   = zs(-df.typical_distance)          # NGAN = duong = co gian cao
df["z_flex"]   = zs(df.pct_flex_payment)           # chon Flex 

print(f"is_urban {df.is_urban.mean():.1%} | hang so tu muc 3: "
      f"BASELINE_MEAN={BASELINE_MEAN}, ZERO_TARGET={ZERO_TARGET}, "
      f"WAKE_RATE={WAKE_RATE}, NB_DISPERSION={NB_DISPERSION}")

is_urban 81.0% | hang so tu muc 3: BASELINE_MEAN=8.0, ZERO_TARGET=0.2, WAKE_RATE=0.25, NB_DISPERSION=4.0


In [9]:
from scipy.stats import nbinom as NB, spearmanr


rng_ab = np.random.default_rng(SEED + 1000)

# Tac dong cua voucher
# cate_true    = tac dong KY VONG theo covariate, khong co nhieu ca nhan
# ite_realized = tac dong THUC TE cua tung rider, co them nhieu ca nhan
tau_cate = (TAU_BASE
            + D_SHORT   * df.z_dist
            + D_MIDDAY  * (df.pref_time_bucket == "midday")
            - P_MORNING * (df.pref_time_bucket == "morning")
            - P_AIRPORT * df.z_air
            + D_PRICE   * df.z_price
            + D_FLEX    * df.z_flex)

tau_noise = rng_ab.standard_normal(len(df))     # nhieu ca nhan
awake     = (~dorm) | woken     # ngu dong khong duoc danh thuc -> voucher vo tac dung

df["cate_true"]    = np.clip(tau_cate, 0, None) * awake
df["ite_realized"] = np.clip(tau_cate + TAU_NOISE * tau_noise, 0, None) * awake

ate_true     = df.cate_true.mean()       # estimand cau truc    = mean(cate_true)
ate_realized = df.ite_realized.mean()    # estimand huu han mau = mean(ite_realized)

# y_base — so chuyen khi KHONG co voucher
mu0    = BASELINE_MEAN * np.exp(B_PRIOR*df.z_prior + B_URBAN*df.z_urban
                                + B_PRICE*df.z_price + B_AIRPORT*df.z_air)
y_base = NB.rvs(NB_DISPERSION, NB_DISPERSION / (NB_DISPERSION + mu0.to_numpy()),
                random_state=rng_ab.integers(1e9))
y_base = np.where(dorm, 0, y_base).astype(np.int32)        # zero inflation

_air = df.pct_airport > df.pct_airport.quantile(0.90)
_flx = df.pct_flex_payment > df.pct_flex_payment.quantile(0.75)
print(f"cate_true    : TB {ate_true:.4f} | SD {df.cate_true.std():.3f} "
      f"| {(df.cate_true == 0).mean():.1%} bang 0")
print(f"ite_realized : TB {ate_realized:.4f} | SD {df.ite_realized.std():.3f} "
      f"| max {df.ite_realized.max():.2f} | {(df.ite_realized == 0).mean():.1%} bang 0")
print(f"   ate_realized - ate_true = {ate_realized - ate_true:+.4f} — lech vi san cat 0: "
      f"nhieu am bi chan lai, nhieu duong thi khong")
print(f"y_base : TB {y_base.mean():.3f} | {(y_base == 0).mean():.1%} bang 0 "
      f"(muc tieu {ZERO_TARGET:.0%})")
print(f"bay    : san bay nen {y_base[_air].mean():.2f} vs {y_base[~_air].mean():.2f} chuyen "
      f"({y_base[_air].mean() / y_base[~_air].mean() - 1:+.0%}) — nhung cate_true "
      f"{df.cate_true[_air].mean():.2f} vs {df.cate_true[~_air].mean():.2f} "
      f"({df.cate_true[_air].mean() / df.cate_true[~_air].mean() - 1:+.0%})")
print(f"Flex   : nhom Flex cao nen {y_base[_flx].mean():.2f} vs {y_base[~_flx].mean():.2f} chuyen "
      f"({y_base[_flx].mean() / y_base[~_flx].mean() - 1:+.0%}) — nhung cate_true "
      f"{df.cate_true[_flx].mean():.2f} vs {df.cate_true[~_flx].mean():.2f} "
      f"({df.cate_true[_flx].mean() / df.cate_true[~_flx].mean() - 1:+.0%}); "
      f"nen KHONG suy ra duoc tu nen")
print(f"kiem   : Spearman(nhieu, total_rides) {spearmanr(tau_noise, df.total_rides).statistic:+.4f}"
      f" | corr(recency, y_base) {np.corrcoef(df.recency_days, y_base)[0, 1]:+.4f}"
      f" | P(y_base>0 | ngu dong) {(y_base[dorm] > 0).mean():.2f}"
      f" | P(woken | ngu dong) {woken[dorm].mean():.3f}")

cate_true    : TB 1.5183 | SD 1.009 | 19.9% bang 0
ite_realized : TB 1.5182 | SD 1.031 | max 7.73 | 20.0% bang 0
   ate_realized - ate_true = -0.0001 — lech vi san cat 0: nhieu am bi chan lai, nhieu duong thi khong
y_base : TB 7.623 | 18.6% bang 0 (muc tieu 20%)
bay    : san bay nen 9.73 vs 7.41 chuyen (+31%) — nhung cate_true 0.18 vs 1.65 (-89%)
Flex   : nhom Flex cao nen 7.12 vs 7.79 chuyen (-9%) — nhung cate_true 1.74 vs 1.45 (+20%); nen KHONG suy ra duoc tu nen
kiem   : Spearman(nhieu, total_rides) +0.0019 | corr(recency, y_base) -0.4911 | P(y_base>0 | ngu dong) 0.00 | P(woken | ngu dong) 0.253


### 5.1 Nhánh RCT — bốc thăm

**Block randomization.** Chia rider thành 10 khối = thập phân vị `total_rides`, rồi bốc đúng 50% **trong từng khối**. `total_rides` là biến dự báo kết quả nền `y_base` mạnh nhất — nó vào `mu0` với hệ số lớn nhất (`B_PRIOR = 0.35`) — nên chia khối theo nó triệt tiêu mất cân bằng ngẫu nhiên

In [10]:
# 10 khoi = thap phan vi total_rides (bien du bao y_base manh nhat)
df["block_id"] = pd.qcut(df.total_rides, 10, labels=False,
                         duplicates="drop").astype(str)

# Sinh treatment RCT boc dung 50% trong tung khoi
df["T_rct"] = 0
for _, g in df.groupby("block_id"):                     # boc dung 50% TRONG TUNG KHOI
    k = int(round(len(g) * TREAT_RATE))
    df.loc[rng_ab.choice(g.index, k, replace=False), "T_rct"] = 1
N_TREATED = int(df.T_rct.sum())

# outcome RCT
df["Y_rct"] = np.rint(y_base + df.ite_realized * df.T_rct).astype(np.int32)

# Tính trung bình nhanh treatment, control và hieu so thuc
m_t, m_c, d_rct = arms(df.Y_rct, df.T_rct)
_lech = df.groupby("block_id").T_rct.mean().sub(TREAT_RATE).abs().max()
print(f"T_rct : {N_TREATED:,} nhan / {len(df) - N_TREATED:,} khong ({df.T_rct.mean():.1%}) | "
      f"{df.block_id.nunique()} khoi, lech ty le lon nhat {_lech:.4f}")
print(f"Y_rct : treatment {m_t:.3f} | control {m_c:.3f} | hieu so {d_rct:+.3f}")

T_rct : 9,999 nhan / 10,001 khong (50.0%) | 10 khoi, lech ty le lon nhat 0.0003
Y_rct : treatment 9.166 | control 7.601 | hieu so +1.566


### 5.2 Nhánh quan sát — marketing tự chọn

`propensity_true` là xác suất được gửi voucher, phụ thuộc **ba confounder** với hệ số trong `TARGETING`. Hằng số `c1` giải bằng `brentq` sao cho tổng số người được gửi bằng đúng nhánh RCT — để hai nhánh so được ở cùng quy mô.

**Confounding** sinh ra: ba biến đó vừa quyết định ai được gửi

In [11]:
# Tạo score quyết định khả năng được nhận voucher
# Trong đó:
# z_prior: rider đi nhiều trước đây.
# z_urban: rider ở khu vực đô thị.
# z_price: rider nhạy giá.
# Ý nghĩa: Nếu rider:đi nhiều, ở Manhattan, nhạy giá thì score lớn → khả năng nhận voucher cao hơn.
lin = (TARGETING["prior"]*df.z_prior + TARGETING["urban"]*df.z_urban
       + TARGETING["price"]*df.z_price).to_numpy()

# c1 chon sao cho so nguoi duoc gui bang DUNG nhanh RCT -> hai nhanh cung quy mo
c1 = brentq(lambda c: (1/(1 + np.exp(-(c + lin)))).sum() - N_TREATED, -20, 20)

# Tạo propensity score dựa trên confounder (z_prior, z_urban, z_price) + hằng số c1
df["propensity_true"] = 1/(1 + np.exp(-(c1 + lin)))

# Sinh treatment observational dựa trên propensity score
df["T_obs"] = (rng_ab.random(len(df)) < df.propensity_true).astype(np.int8)

# outcome observational
df["Y_obs"] = np.rint(y_base + df.ite_realized * df.T_obs).astype(np.int32)

# Tính trung bình nhanh treatment, control và hieu so thuc
m_t, m_c, d_obs = arms(df.Y_obs, df.T_obs)
_, _, d_rct     = arms(df.Y_rct, df.T_rct)
print(f"T_obs : {int(df.T_obs.sum()):,} nhan ({df.T_obs.mean():.1%}) | propensity "
      f"[{df.propensity_true.min():.3f}, {df.propensity_true.max():.3f}]")
print(f"Y_obs : treatment {m_t:.3f} | control {m_c:.3f} | hieu so {d_obs:+.3f}")
print()
print(f"ate_true {ate_true:.4f} | ate_realized {ate_realized:.4f} (estimand cua RCT)")
print(f"  ->  RCT {d_rct:+.3f} (lech {d_rct / ate_realized - 1:+.0%}, co boc tham)  |  "
      f"OBS {d_obs:+.3f} (lech {d_obs / ate_realized - 1:+.0%}, confounding)")

T_obs : 9,904 nhan (49.5%) | propensity [0.001, 0.947]
Y_obs : treatment 10.665 | control 6.332 | hieu so +4.333

ate_true 1.5183 | ate_realized 1.5182 (estimand cua RCT)
  ->  RCT +1.566 (lech +3%, co boc tham)  |  OBS +4.333 (lech +185%, confounding)


## 6. Xuất kết quả

In [ ]:
EXP_COLS = ["user_id", "age", "gender", "home_borough", "total_rides", "recency_days",
            "typical_distance", "route_entropy", "pct_airport", "pref_time_bucket",
            "weekend_ratio", "pct_flex_payment", "pct_tip_rate", "avg_fare",
            "is_urban", "block_id", "T_rct", "T_obs", "Y_rct", "Y_obs",
            "cate_true", "ite_realized", "propensity_true"]
df[EXP_COLS].to_csv(os.path.join(OUTDIR, "experiment_ab_final.csv"), index=False)

for fn in sorted(os.listdir(OUTDIR)):
    print(f"  {fn:<32} {os.path.getsize(os.path.join(OUTDIR, fn)):>10,} bytes")
print(f"\n{len(df):,} rider | T_rct {df.T_rct.sum():,} | T_obs {df.T_obs.sum():,}")
print(f"ate_true     = {ate_true:.4f} chuyen / rider / {W_OUT} ngay  (= trung binh cot cate_true)")
print(f"ate_realized = {ate_realized:.4f} chuyen / rider / {W_OUT} ngay  (= trung binh cot ite_realized) — day la estimand cua nhanh RCT")

  customer_features_final.csv       1,776,077 bytes
  experiment_ab_final.csv           3,044,589 bytes

20,000 rider | T_rct 9,999 | T_obs 9,904
ate_true     = 1.5183 chuyen / rider / 30 ngay  (= trung binh cot cate_true)
ate_realized = 1.5182 chuyen / rider / 30 ngay  (= trung binh cot ite_realized) — day la estimand cua nhanh RCT
